# Environment Setup

In [1]:
import sys, os, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working directory:', os.getcwd())
!pip install -q --force-reinstall "torch==2.10.0" --index-url https://download.pytorch.org/whl/cu128

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Working directory: /content
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.10.0+cu128 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.7.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cuda-python 12.9.7 requires cuda-bindings~=12.9.7, but you have cuda-bindings 12.9.4 which is incompatible.


In [2]:
import torch, shutil
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
nvcc = shutil.which('nvcc')
print('nvcc:', nvcc if nvcc else 'not found')

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4
nvcc: /usr/local/cuda/bin/nvcc


# Clone Repo

In [3]:
import os
REPO_URL  = 'https://github.com/jeromereddy9/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems'
REPO_PATH = '/content/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems'
if not os.path.exists(REPO_PATH):
    os.system(f'git clone {REPO_URL}')
os.chdir(REPO_PATH)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

Working directory: /content/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems
Contents: ['src', '.git', '.gitmodules', '.idea', '.gitignore', 'LICENSE']


# Install Dependencies

In [4]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'numpy==1.26.4', 'recbole==1.2.0'], check=True)
print('recbole installed')

recbole installed


In [5]:
import torch
!pip install -U pip setuptools wheel
!pip install ninja packaging
!pip install causal-conv1d --no-build-isolation
import causal_conv1d
print("causal-conv1d:", causal_conv1d.__version__)
if torch.cuda.is_available():
    print("GPU detected — installing Mamba dependencies...")
    !pip install mamba-ssm==2.3.1 --no-build-isolation
    import mamba_ssm
    print("mamba-ssm version:", mamba_ssm.__version__)
    print("causal-conv1d installed successfully")

else:
    print("No GPU — skipping mamba-ssm")

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 78.1.0
    Uninstalling setuptools-78.1.0:
      Successfully uninstalled setuptools-78.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


  Preparing metadata (pyproject.toml) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.2.post1-cp312-cp312-linux_x86_64.whl size=193835395 sha256=c16c1c48d4fa63415cc797e02d69f97248c57c04627d99e394d5bb0ef266e288
  Stored in directory: /root/.cache/pip/wheels/ea/f0/26/5d87ae05a302e6dc8016c50cd8c7ee779585f593b9580e7cf8
Successfully built causal-conv1d
causal-conv1d: 1.6.2.post1
GPU detected — installing Mamba dependencies...
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl size=533592144 sha256=d66a3c4c94a693e02d341cace7a6af0b72177b6afa655a25e3a6505130a68cbf
  Stored in directory: /root/.cache/pip/wheels/28/83/54/d45107838fec575b93f5d723f56351cee19a1b13bcd4ec9f3f
Successfully built mamba-ssm
mamba-ssm version: 2.3.1
causal-conv1d installed successfully


# Mount Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Copy Datasets from Drive

In [7]:
import os, shutil

DRIVE_DATASET_ROOT   = '/content/drive/MyDrive/preprocessed'
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/checkpoints'
DRIVE_RESULTS_DIR    = '/content/drive/MyDrive/results'
LOCAL_DATASET_ROOT   = os.path.join(REPO_PATH, 'src/datasets/preprocessed')

os.makedirs(LOCAL_DATASET_ROOT,   exist_ok=True)
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR,    exist_ok=True)

DATASETS = [
    'amazon_videogames',
    'amazon_toys_and_games',
    'movielens_1m',
    'lastfm_1k',
]

for dataset in DATASETS:
    src  = os.path.join(DRIVE_DATASET_ROOT, dataset)
    dest = os.path.join(LOCAL_DATASET_ROOT, dataset)
    if not os.path.exists(dest):
        shutil.copytree(src, dest)
        print(f'Copied: {dataset}')
    else:
        print(f'Already exists: {dataset}')

print('\nCheckpoint dir:', DRIVE_CHECKPOINT_DIR)
print('Results dir   :', DRIVE_RESULTS_DIR)

Copied: amazon_videogames
Copied: amazon_toys_and_games
Copied: movielens_1m
Copied: lastfm_1k

Checkpoint dir: /content/drive/MyDrive/checkpoints
Results dir   : /content/drive/MyDrive/results


# Imports

In [26]:
import sys, os
sys.path.insert(0, REPO_PATH)

import warnings, logging, traceback, csv, pickle, time, json
from datetime import datetime
warnings.filterwarnings('ignore')
logging.getLogger('recbole').setLevel(logging.ERROR)

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger
from logging import getLogger

from src.utils import path_builder
from src.models.Baselines.GRU4Rec import GRU4Rec
from src.models.Baselines.SASRec import SASRec
from src.models.Baselines.CL4SRec import CL4SRec
from src.models.Baselines.DouRec import DuoRec
from src.models.Baselines.mamba4rec import Mamba4Rec
from src.models.Baselines.gated_mamba import SIGMA
from src.models.SSM_CL.mamba4rec_cl import Mamba4Rec_CL
from src.models.SSM_CL.SIGMA_cl import SIGMA_CL

print('All imports successful')

All imports successful


# Stage Selection
Set STAGE before running the training cell.
- `'1'`  — Base SSMs: Mamba4Rec CE/BPR + SIGMA CE/BPR
- `'2'` — Mamba4Rec_CL CE: InfoNCE + DCL
- `'3'` — Mamba4Rec_CL BPR: InfoNCE + DCL
- `'4'` — SIGMA_CL CE: InfoNCE + DCL
- `'5'` — SIGMA_CL BPR: InfoNCE + DCL
- `'6'`  — Baselines: GRU4Rec, SASRec, CL4SRec, DuoRec

In [27]:
STAGE = '1'

CONFIG_DIR        = path_builder('src/configs')
DATASET_CONFIG    = path_builder(CONFIG_DIR + '/dataset.yaml')
TRAINING_CONFIG   = path_builder(CONFIG_DIR + '/training.yaml')
MODELS_CONFIG_DIR = path_builder(CONFIG_DIR + '/models')
CSV_PATH          = os.path.join(DRIVE_RESULTS_DIR, f'stage{STAGE}_results.csv')

STAGE_EXPERIMENTS = {
    '1': [
        (Mamba4Rec, 'Mamba4Rec', 'mamba4rec', 'CE',  None),
        (Mamba4Rec, 'Mamba4Rec', 'mamba4rec', 'BPR', None),
        (SIGMA,     'SIGMA',     'sigma',     'CE',  None),
        (SIGMA,     'SIGMA',     'sigma',     'BPR', None),
    ],
    '2': [
        (Mamba4Rec_CL, 'Mamba4Rec_CL', 'mamba4rec_cl', 'CE', 'info_nce'),
        (Mamba4Rec_CL, 'Mamba4Rec_CL', 'mamba4rec_cl', 'CE', 'dcl'),
    ],
    '3': [
        (Mamba4Rec_CL, 'Mamba4Rec_CL', 'mamba4rec_cl', 'BPR', 'info_nce'),
        (Mamba4Rec_CL, 'Mamba4Rec_CL', 'mamba4rec_cl', 'BPR', 'dcl'),
    ],
    '4': [
        (SIGMA_CL, 'SIGMA_CL', 'sigma_cl', 'CE', 'info_nce'),
        (SIGMA_CL, 'SIGMA_CL', 'sigma_cl', 'CE', 'dcl'),
    ],
    '5': [
        (SIGMA_CL, 'SIGMA_CL', 'sigma_cl', 'BPR', 'info_nce'),
        (SIGMA_CL, 'SIGMA_CL', 'sigma_cl', 'BPR', 'dcl'),
    ],
    '6': [
        (GRU4Rec, 'GRU4Rec', 'gru4rec', None, None),
        (SASRec,  'SASRec',  'sasrec',  None, None),
        (DuoRec,  'DuoRec',  'duorec',  None, None),
        (CL4SRec, 'CL4SRec', 'cl4srec', None, None),
    ],
}

STAGE_LABELS = {
    '1':  'Base SSMs (Mamba4Rec + SIGMA)',
    '2': 'Mamba4Rec_CL CE (InfoNCE + DCL)',
    '3': 'Mamba4Rec_CL BPR (InfoNCE + DCL)',
    '4': 'SIGMA_CL CE (InfoNCE + DCL)',
    '5': 'SIGMA_CL BPR (InfoNCE + DCL)',
    '6':  'Baselines',
}

assert STAGE in STAGE_EXPERIMENTS, f'STAGE must be one of {list(STAGE_EXPERIMENTS.keys())}'
EXPERIMENTS = STAGE_EXPERIMENTS[STAGE]

print(f'Stage        : {STAGE} — {STAGE_LABELS[STAGE]}')
print(f'Experiments  : {len(EXPERIMENTS)} per dataset')
print(f'Datasets     : {DATASETS}')
print(f'Total        : {len(EXPERIMENTS) * len(DATASETS)}')
print(f'CSV          : {CSV_PATH}')
print(f'Checkpoints  : {DRIVE_CHECKPOINT_DIR}')

Stage        : 1 — Base SSMs (Mamba4Rec + SIGMA)
Experiments  : 4 per dataset
Datasets     : ['amazon_videogames', 'amazon_toys_and_games', 'movielens_1m', 'lastfm_1k']
Total        : 16
CSV          : /content/drive/MyDrive/results/stage1_results.csv
Checkpoints  : /content/drive/MyDrive/checkpoints


### Geometry Analysis

In [28]:
def compute_isotropy(embeddings: torch.Tensor):
    embeddings = embeddings - embeddings.mean(dim=0)
    cov = torch.mm(embeddings.T, embeddings) / embeddings.shape[0]
    eigenvalues = torch.linalg.eigvalsh(cov).clamp(min=1e-10)
    eigenvalues = eigenvalues / eigenvalues.sum()
    entropy = -(eigenvalues * torch.log(eigenvalues)).sum()
    return (torch.exp(entropy) / embeddings.shape[1]).item()

def compute_effective_rank(embeddings: torch.Tensor):
    embeddings = embeddings - embeddings.mean(dim=0)
    _, singular_values, _ = torch.linalg.svd(embeddings, full_matrices=False)
    singular_values = singular_values.clamp(min=1e-10)
    singular_values = singular_values / singular_values.sum()
    entropy = -(singular_values * torch.log(singular_values)).sum()
    return torch.exp(entropy).item()

def extract_embeddings(model, n_items: int):
    with torch.no_grad():
        return model.item_embedding.weight[:n_items].cpu().float()

### Custom Trainer with Timing

In [29]:
class TimedTrainer(Trainer):
    def __init__(self, config, model):
        super().__init__(config, model)
        self.epoch_times = []
        self.train_loss_history = []
        self.valid_metric_history = []
        self.total_train_time = 0

    def fit(self, train_data, valid_data=None, saved=True, show_progress=True):
        self.total_train_time = 0
        self.epoch_times = []
        self.train_loss_history = []
        self.valid_metric_history = []
        best_score, best_result = super().fit(
            train_data, valid_data=valid_data, saved=saved, show_progress=show_progress
        )
        return best_score, best_result

    def _train_epoch(self, train_data, epoch_idx, loss_func=None, show_progress=True):
        epoch_start = time.perf_counter()
        loss = super()._train_epoch(train_data, epoch_idx, loss_func=loss_func, show_progress=show_progress)
        epoch_time = time.perf_counter() - epoch_start

        self.epoch_times.append(epoch_time)
        self.train_loss_history.append(loss)
        self.total_train_time += epoch_time

        if hasattr(self, 'best_valid_score') and self.best_valid_score is not None:
            self.valid_metric_history.append(self.best_valid_score)

        return loss


### Experiment Runner

In [30]:
def make_exp_id(model_name, loss_type, cl_loss_type, dataset):
    parts = [model_name]
    if loss_type:    parts.append(loss_type)
    if cl_loss_type: parts.append(cl_loss_type)
    parts.append(dataset)
    return '_'.join(parts)



In [31]:
def run_unified_experiment(model_class, model_name, config_file, dataset_name,
                           loss_type=None, cl_loss_type=None, skip_if_exists=True):
    exp_id = make_exp_id(model_name, loss_type, cl_loss_type, dataset_name)
    checkpoint_path = os.path.join(DRIVE_CHECKPOINT_DIR, f'{exp_id}.pkl')

    if skip_if_exists and os.path.exists(checkpoint_path):
        print(f"{exp_id} exists - skipping")
        return None

    print(f"  {exp_id}")
    print(f"  Started: {datetime.now().strftime('%H:%M:%S')}")

    result = {
        'exp_id': exp_id, 'model': model_name, 'loss_type': loss_type or 'default',
        'cl_loss_type': cl_loss_type or 'none', 'dataset': dataset_name,
        'status': 'failed', 'error': '',
        'total_train_time_sec': None, 'total_train_time_min': None, 'avg_epoch_time_sec': None,
        'num_epochs': None, 'early_stopped': None, 'best_epoch': None,
        'best_valid_score': None, 'hit@5': None, 'hit@10': None, 'hit@20': None,
        'ndcg@5': None, 'ndcg@10': None, 'ndcg@20': None, 'mrr@5': None, 'mrr@10': None, 'mrr@20': None,
        'isotropy': None, 'effective_rank': None, 'embedding_dim': None, 'n_items': None,
        'epoch_times': [], 'train_losses': [], 'valid_metrics': [],
    }

    try:
        config_dict = {
            'data_path': LOCAL_DATASET_ROOT
        }
        if loss_type:     config_dict['loss_type'] = loss_type
        if cl_loss_type:  config_dict['cl_loss_type'] = cl_loss_type
        if loss_type == 'BPR':
            config_dict['train_neg_sample_args'] = {
                'distribution': 'uniform', 'sample_num': 1,
                'alpha': 1.0, 'dynamic': False, 'candidate_num': 0
            }

        config = Config(
            model=model_class,
            dataset=dataset_name,
            config_file_list=[
                DATASET_CONFIG, TRAINING_CONFIG, path_builder(MODELS_CONFIG_DIR + f'/{config_file}.yaml'),
            ],
            config_dict=config_dict,
        )

        init_seed(config['seed'], config['reproducibility'])
        init_logger(config)

        print(f"  Loading dataset & Preparing data ...")
        dataset = create_dataset(config)
        train_data, valid_data, test_data = data_preparation(config, dataset)

        model = model_class(config, dataset).to(config['device'])
        trainer = TimedTrainer(config, model)

        print(f"  Starting training...")
        train_start = time.perf_counter()
        best_valid_score, best_valid_result = trainer.fit(train_data, valid_data, saved=True, show_progress=True)
        total_train_time = time.perf_counter() - train_start

        epoch_times = trainer.epoch_times
        train_losses = trainer.train_loss_history
        valid_metrics = trainer.valid_metric_history
        best_epoch = getattr(trainer, 'best_valid_epoch', len(epoch_times) - 1)
        early_stopped = len(epoch_times) < config['epochs']

        epoch_history = [
            {'epoch': i, 'train_loss': l, 'valid_score': valid_metrics[i] if i < len(valid_metrics) else None}
            for i, l in enumerate(train_losses)
        ]

        with open(checkpoint_path, 'wb') as f:
            pickle.dump({
                'model_state_dict': trainer.model.state_dict(),
                'config': {k: v for k, v in config.final_config_dict.items()},
                'best_valid_score': best_valid_score,
                'best_valid_result': best_valid_result,
                'best_epoch': best_epoch, 'total_epochs': len(epoch_times),
                'early_stopped': early_stopped, 'epoch_history': epoch_history,
                'epoch_times': epoch_times, 'total_train_time': total_train_time,
                'exp_id': exp_id,
            }, f)

        print(f"\n  Evaluating on test set...")
        test_result = trainer.evaluate(test_data, load_best_model=False, show_progress=True)

        print(f"  Computing geometry...")
        embeddings = extract_embeddings(model, config['n_items'] if 'n_items' in config else dataset.item_num)
        isotropy = compute_isotropy(embeddings)
        eff_rank = compute_effective_rank(embeddings)

        result.update({
            'status': 'success', 'total_train_time_sec': total_train_time,
            'total_train_time_min': total_train_time / 60,
            'avg_epoch_time_sec': np.mean(epoch_times) if epoch_times else None,
            'num_epochs': len(epoch_times), 'early_stopped': early_stopped, 'best_epoch': best_epoch,
            'best_valid_score': best_valid_score,
            'hit@5': test_result.get('hit@5'), 'hit@10': test_result.get('hit@10'), 'hit@20': test_result.get('hit@20'),
            'ndcg@5': test_result.get('ndcg@5'), 'ndcg@10': test_result.get('ndcg@10'), 'ndcg@20': test_result.get('ndcg@20'),
            'mrr@5': test_result.get('mrr@5'), 'mrr@10': test_result.get('mrr@10'), 'mrr@20': test_result.get('mrr@20'),
            'isotropy': isotropy, 'effective_rank': eff_rank,
            'embedding_dim': embeddings.shape[1], 'n_items': embeddings.shape[0],
            'epoch_times': epoch_times, 'train_losses': train_losses, 'valid_metrics': valid_metrics,
        })

        print(f"\n  Complete")
        print(f"  NDCG@10: {result['ndcg@10']:.4f} | Hit@10: {result['hit@10']:.4f}")
        print(f"  Isotropy: {result['isotropy']:.4f} | Eff. Rank: {result['effective_rank']:.2f}/{result['embedding_dim']}")
        print(f"  Training: {result['total_train_time_min']:.1f} min ({result['avg_epoch_time_sec']:.0f}s/epoch)")
    except Exception as e:
        result['error'] = str(e)
        print(f"  FAILED: {e}")
        traceback.print_exc()

    return result


### CSV Helpers & Plotting Definitions

In [33]:
RESULTS_HEADERS = [
    'exp_id', 'model', 'loss_type', 'cl_loss_type', 'dataset', 'status', 'error',
    'total_train_time_min', 'avg_epoch_time_sec', 'num_epochs', 'early_stopped', 'best_epoch', 'best_valid_score',
    'hit@5', 'hit@10', 'hit@20', 'ndcg@5', 'ndcg@10', 'ndcg@20', 'mrr@5', 'mrr@10', 'mrr@20',
    'isotropy', 'effective_rank', 'embedding_dim', 'n_items',
    'train_losses', 'epoch_times', 'valid_metrics',
]

def save_results(results_to_save, path):

    if not isinstance(results_to_save, list):
        results_to_save = [results_to_save]

    file_exists = os.path.exists(path)

    with open(path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_HEADERS)
        if not file_exists:
            writer.writeheader()

        for r in results_to_save:
            if not r: continue
            row = {}
            for k in RESULTS_HEADERS:
                if k in r:
                    if k in ['train_losses', 'epoch_times', 'valid_metrics']:

                        row[k] = json.dumps(r[k]) if r[k] is not None else ''
                    elif r[k] is not None:
                        row[k] = r[k]
                    else:
                        row[k] = ''
                else:
                    row[k] = ''
            writer.writerow(row)
    print(f"\nResults saved/appended to: {path}")

DATASET_LABELS = {'amazon_videogames': 'Video Games', 'amazon_toys_and_games': 'Toys & Games', 'movielens_1m': 'ML-1M', 'lastfm_1k': 'LastFM-1K'}
MODEL_COLORS = {
    'Mamba4Rec_CE': '#F59E0B', 'Mamba4Rec_BPR': '#D97706', 'Mamba4Rec_CL_CE_info_nce': '#EF4444', 'Mamba4Rec_CL_CE_dcl': '#DC2626',
    'SIGMA_CE': '#10B981', 'SIGMA_BPR': '#059669', 'SIGMA_CL_CE_info_nce': '#34D399', 'SIGMA_CL_CE_dcl': '#6EE7B7',
    'GRU4Rec': '#6B7280', 'SASRec': '#3B82F6', 'CL4SRec': '#06B6D4', 'DuoRec': '#8B5CF6',
}

def get_color(label): return MODEL_COLORS.get(label, '#9CA3AF')
def model_label(row):
    parts = [row['model']]
    if row['loss_type'] != 'default': parts.append(row['loss_type'])
    if row['cl_loss_type'] != 'none': parts.append(row['cl_loss_type'])
    return '_'.join(parts)

def plot_convergence(results, output_dir):
    successful = [r for r in results if r and r['status'] == 'success' and r.get('train_losses')]
    for r in successful:
        exp_id = r['exp_id']
        epochs = list(range(len(r['train_losses'])))
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(epochs, r['train_losses'], 'b-', linewidth=1.5)
        ax1.set_title(f'{exp_id} — Training Loss')
        if r.get('valid_metrics'):
            ax2.plot(list(range(len(r['valid_metrics']))), r['valid_metrics'], 'g-', linewidth=1.5)
            ax2.axhline(y=r.get('best_valid_score', 0), color='r', linestyle='--')
        ax2.set_title(f'{exp_id} — Validation')
        plt.savefig(os.path.join(output_dir, f'convergence_{exp_id}.png'), dpi=100, bbox_inches='tight')
        plt.close()

def plot_metric_comparison(results, metric, title, output_path):
    successful = [r for r in results if r and r['status'] == 'success' and r.get(metric) is not None]
    if not successful: return
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(20, 6), sharey=True)
    if len(DATASETS) == 1: axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    for ax, dataset in zip(axes, DATASETS):
        ds_results = [r for r in successful if r['dataset'] == dataset]
        if not ds_results: ax.set_visible(False); continue
        labels = [model_label(r) for r in ds_results]
        values = [float(r[metric]) for r in ds_results]
        bars = ax.bar(range(len(labels)), values, color=[get_color(l) for l in labels])
        ax.set_title(DATASET_LABELS.get(dataset, dataset))
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()

def plot_timing_comparison(results, output_path):
    plot_metric_comparison(results, 'total_train_time_min', 'Training Time by Model and Dataset', output_path)

# Run Training and Evaluation

In [25]:

plots_dir = os.path.join(DRIVE_RESULTS_DIR, 'plots')
os.makedirs(plots_dir, exist_ok=True)


print(f"  Training & Evaluation")
print(f"  Stage {STAGE}: {STAGE_LABELS[STAGE]}")
print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

current_run_results = []
global_start = time.perf_counter()

csv_path = os.path.join(DRIVE_RESULTS_DIR, f'stage{STAGE}_results_full.csv')

for dataset in DATASETS:
    print(f"\n  DATASET: {dataset}\n")
    for model_class, model_name, config_file, loss_type, cl_loss_type in EXPERIMENTS:
        result = run_unified_experiment(
            model_class=model_class,
            model_name=model_name,
            config_file=config_file,
            dataset_name=dataset,
            loss_type=loss_type,
            cl_loss_type=cl_loss_type,
            skip_if_exists=True
        )
        if result:
            current_run_results.append(result)
            save_results(result, csv_path)

total_time = (time.perf_counter() - global_start) / 60



expected_total_experiments = len(EXPERIMENTS) * len(DATASETS)

all_results_from_csv = []
if os.path.exists(csv_path):
    with open(csv_path, 'r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            for key in ['total_train_time_min', 'avg_epoch_time_sec', 'num_epochs', 'best_epoch', 'best_valid_score',
                        'hit@5', 'hit@10', 'hit@20', 'ndcg@5', 'ndcg@10', 'ndcg@20', 'mrr@5', 'mrr@10', 'mrr@20',
                        'isotropy', 'effective_rank', 'embedding_dim', 'n_items']:
                if key in row and row[key] not in ['None', '']:
                    try:
                        row[key] = float(row[key])
                    except ValueError:
                        pass
            # Deserialize list-like fields from JSON strings
            for list_key in ['train_losses', 'epoch_times', 'valid_metrics']:
                if list_key in row and row[list_key]:
                    try:
                        row[list_key] = json.loads(row[list_key])
                    except json.JSONDecodeError:
                        row[list_key] = [] # Set to empty list if parsing fails
                else:
                    row[list_key] = [] # Ensure it's an empty list if not present or empty string
            all_results_from_csv.append(row)

if len(all_results_from_csv) == expected_total_experiments:
    print(f"\nAll {expected_total_experiments} experiments for Stage {STAGE} are complete. Generating plots...")
    plot_convergence(all_results_from_csv, plots_dir)
    plot_metric_comparison(all_results_from_csv, 'ndcg@10', 'NDCG@10 by Model and Dataset', os.path.join(plots_dir, f'stage{STAGE}_ndcg10.png'))
    plot_metric_comparison(all_results_from_csv, 'hit@10', 'Hit@10 by Model and Dataset', os.path.join(plots_dir, f'stage{STAGE}_hit10.png'))
    plot_metric_comparison(all_results_from_csv, 'mrr@10', 'MRR@10 by Model and Dataset', os.path.join(plots_dir, f'stage{STAGE}_mrr10.png'))
    plot_timing_comparison(all_results_from_csv, os.path.join(plots_dir, f'stage{STAGE}_training_time.png'))
else:
    print(f"\nOnly {len(all_results_from_csv)}/{expected_total_experiments} experiments for Stage {STAGE} are complete. Skipping plot generation until all are done.")

print(f"\n  Results: {csv_path}")
print(f"  Plots: {plots_dir}")

  Training & Evaluation
  Stage 1: Base SSMs (Mamba4Rec + SIGMA)
  Started: 2026-08-17 19:05:35

  DATASET: amazon_videogames

  Mamba4Rec_CE_amazon_videogames
  Started: 19:05:35


  Loading dataset & Preparing data ...
  Starting training...


Evaluate   : 100%|█████████████████████████| 13/13 [00:00<00:00, 15.83it/s, GPU RAM: 2.72 G/22.03 G]



  Evaluating on test set...


Evaluate   : 100%|█████████████████████████| 13/13 [00:00<00:00, 15.74it/s, GPU RAM: 2.72 G/22.03 G]


  Computing geometry...

  Complete
  NDCG@10: 0.0507 | Hit@10: 0.0983
  Isotropy: 0.8019 | Eff. Rank: 60.56/64
  Training: 28.3 min (12s/epoch)


ValueError: dict contains fields not in fieldnames: 'train_losses', 'total_train_time_sec', 'epoch_times', 'valid_metrics'